# Read a BSON file in Python with chDB

Companion to [read-bson-file-python](https://clickhouse.com/resources/engineering/read-bson-file-python).

Run `./generate.sh` first to create `data/`. Requires `pip install chdb pandas` (and `pip install pymongo` for the perf contrast).

## 1. Read a BSON file into a DataFrame (schema auto-inferred)

In [ ]:
from chdb.datastore import DataStore

df = DataStore.from_file("data/events.bson", format="BSONEachRow")
df

In [ ]:
df.dtypes

## 2. Filter + aggregate the way you already do (pandas, not SQL)

In [ ]:
purchases = df[df["event_type"] == "purchase"].groupby("country")["revenue"].sum()
purchases.to_pandas().round(2)

## 3. Hand off to real pandas when you need it

In [ ]:
pdf = df.to_pandas()   # a real pandas.DataFrame, in memory
type(pdf)

## 4. Performance: chDB DataStore vs pymongo decode + pandas on a 2M-row BSON

Apple M4 Pro (14 cores, 24 GB RAM, macOS), best-of-3 warm: ~43x faster than the pymongo decode path.

In [ ]:
import time

def datastore_agg():
    d = DataStore.from_file("data/events_large.bson", format="BSONEachRow")
    r = (d[d["event_type"] == "purchase"]
         .groupby("country")["revenue"]
         .sum()
         .sort_values(ascending=False))
    return r.to_pandas()

def pymongo_pandas_agg():
    import pandas as pd
    from bson import decode_file_iter
    with open("data/events_large.bson", "rb") as f:
        pdf = pd.DataFrame(decode_file_iter(f))
    p = pdf[pdf.event_type == b"purchase"]
    return (p.groupby("country")["revenue"]
             .sum()
             .sort_values(ascending=False))

def best_of_3(fn):
    fn()
    best = float("inf")
    for _ in range(3):
        t0 = time.perf_counter(); fn(); best = min(best, time.perf_counter() - t0)
    return best

py_s = best_of_3(pymongo_pandas_agg)
ds_s = best_of_3(datastore_agg)
print(f"pymongo decode + pandas:        {py_s:.3f}s")
print(f"import chdb.datastore:          {ds_s:.3f}s")
print(f"speedup:                        {py_s / ds_s:.1f}x")